# Code Vulnerability Classification

In [73]:
import sys
print(sys.executable)

c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [ ]:
import sys
!{sys.executable} -m pip install xgboost # Installed XGBoost library in the write environment


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [99]:
import pandas as pd
import numpy as np
import re   # For cleaning text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, precision_score, f1_score
from sklearn.model_selection import cross_val_score
from scipy.sparse import hstack  # To combine features

## Importing the dataset

In [76]:

train_df = pd.read_csv("data_C_Ours_train.csv") # training dataset

test_df = pd.read_csv("data_C_Ours_test.csv")   # testing dataset

## inspecting the data

In [77]:
print("Training Data Sample:")
train_df.head()


Training Data Sample:


,Unnamed: 0,cwe_id,code,repo,path,url,sha,target
0,3530,NVD-CWE-Other,"static int ras_getdatastd(jas_stream_t *in, ra...",visit repo url,src/libjasper/ras/ras_dec.c,https://github.com/mdadams/jasper,106829516887531,1
1,4682,CWE-732,char *M_fs_path_tmpdir(M_fs_system_t sys_type)...,visit repo url,base/fs/m_fs_path.c,https://github.com/Monetra/mstdlib,88247877163772,1
2,4537,['CWE-20'],"static int make_indexed_dir(handle_t *handle, ...",linux-2.6,NaN,NaN,287883023626102361284836105016297785515,0
3,2148,CWE-476,struct btrfs_device *btrfs_find_device(struct ...,visit repo url,fs/btrfs/volumes.c,https://github.com/torvalds/linux,239983200436702,1
4,4426,['CWE-264'],"ssize_t sock_no_sendpage(struct socket *sock, ...",linux-2.6,NaN,NaN,78434298454054858077706649493150329771,0


In [78]:
print("Testing Data Sample:")
test_df.head()

Testing Data Sample:


,Unnamed: 0,cwe_id,code,repo,path,url,sha,target
0,1159,CWE-264,"SYSCALL_DEFINE3(osf_sysinfo, int, command, cha...",visit repo url,arch/alpha/kernel/osf_sys.c,https://github.com/torvalds/linux,17547188397697,1
1,3825,['CWE-120'],struct usb_host_endpoint *uvc_find_endpoint(st...,linux-2.6,NaN,NaN,243378003459452140277809791624786448709,0
2,3225,['CWE-189'],static int jpc_dec_cp_setfromqcx(jpc_dec_cp_t ...,jasper,NaN,NaN,212650590176928811121263296586304180677,0
3,4232,CWE-78,"static int r_core_cmd_subst_i(RCore *core, cha...",visit repo url,libr/core/cmd.c,https://github.com/radareorg/radare2,4710862966549,1
4,518,CWE-119,static void coerce_reg_to_32(struct bpf_reg_st...,visit repo url,kernel/bpf/verifier.c,https://github.com/torvalds/linux,163420304807872,1


In [79]:
# Checking shape (rows, columns)
print("\nTrain Shape:", train_df.shape)
print("Test Shape:", test_df.shape)


Train Shape: (10826, 8)
Test Shape: (2706, 8)


In [80]:
# Checking data types
print("\nData Types:")
print(train_df.dtypes)

# Checking missing values
print("\nMissing Values:")
print(train_df.isnull().sum())

# Checking target distribution to understand class imbalance
print("\nTarget Distribution:")
print(train_df['target'].value_counts())


Data Types:
Unnamed: 0     int64
cwe_id           str
code             str
repo             str
path             str
url              str
sha           object
target         int64
dtype: object

Missing Values:
Unnamed: 0       0
cwe_id           0
code             0
repo             0
path          5413
url           5413
sha              0
target           0
dtype: int64

Target Distribution:
target
1    5413
0    5413
Name: count, dtype: int64


## Data Cleaning

In [81]:
# List of columns to be removed
columns_to_drop = ['repo', 'path', 'url', 'sha', 'Unnamed: 0'] # dropped path because major paths are mising and it is not useful for prediction

In [82]:
# Dropping columns one by one present in the list from both train and test datasets
for col in columns_to_drop:
    if col in train_df.columns:
        train_df = train_df.drop(columns=col)
        
    if col in test_df.columns:
        test_df = test_df.drop(columns=col)

In [83]:
# Checking remaining columns
print("Remaining columns in train data:")
print(train_df.columns)

print("\nRemaining columns in test data:")
print(test_df.columns)

Remaining columns in train data:
Index(['cwe_id', 'code', 'target'], dtype='str')

Remaining columns in test data:
Index(['cwe_id', 'code', 'target'], dtype='str')


In [84]:
# Function to clean code snippets

def clean_code(code):
    code = str(code)          # to convert it into string
    code = code.lower()       # converting to lowercase
    
    # removing comments 
    code = re.sub(r'//.*|/\*.*?\*/|#.*', '', code)
    
    # removing extra spaces
    code = re.sub(r'\s+', ' ', code)
    
    return code.strip()



In [85]:
# Apply cleaning to train data
train_df['clean_code'] = train_df['code'].apply(clean_code)

# Apply cleaning to test data
test_df['clean_code'] = test_df['code'].apply(clean_code)

# checking original and cleaned code for the first row
print("Original Code:\n", train_df['code'].iloc[0])
print("\nCleaned Code:\n", train_df['clean_code'].iloc[0])

Original Code:
 static int ras_getdatastd(jas_stream_t *in, ras_hdr_t *hdr, ras_cmap_t *cmap,
  jas_image_t *image)
{
	int pad;
	int nz;
	int z;
	int c;
	int y;
	int x;
	int v;
	int i;
	jas_matrix_t *data[3];
	cmap = 0;
	for (i = 0; i < jas_image_numcmpts(image); ++i) {
		data[i] = jas_matrix_create(1, jas_image_width(image));
		assert(data[i]);
	}
	pad = RAS_ROWSIZE(hdr) - (hdr->width * hdr->depth + 7) / 8;
	for (y = 0; y < hdr->height; y++) {
		nz = 0;
		z = 0;
		for (x = 0; x < hdr->width; x++) {
			while (nz < hdr->depth) {
				if ((c = jas_stream_getc(in)) == EOF) {
					return -1;
				}
				z = (z << 8) | c;
				nz += 8;
			}
			v = (z >> (nz - hdr->depth)) & RAS_ONES(hdr->depth);
			z &= RAS_ONES(nz - hdr->depth);
			nz -= hdr->depth;
			if (jas_image_numcmpts(image) == 3) {
				jas_matrix_setv(data[0], x, (RAS_GETRED(v)));
				jas_matrix_setv(data[1], x, (RAS_GETGREEN(v)));
				jas_matrix_setv(data[2], x, (RAS_GETBLUE(v)));
			} else {
				jas_matrix_setv(data[0], x, (v));
			}

In [86]:
# Function to convert cwe_id into number

def clean_cwe(cwe):
    cwe = str(cwe)  # make sure it's string
    
    # extract numbers from string
    numbers = re.findall(r'\d+', cwe)
    
    # if number found → return it
    if len(numbers) > 0:
        return int(numbers[0])
    else:
        return 0   # for cases like 'NVD-CWE-Other'

In [87]:
# Applying cleaning function to train data
train_df['cwe_clean'] = train_df['cwe_id'].apply(clean_cwe)

# Applying cleaning function to test data
test_df['cwe_clean'] = test_df['cwe_id'].apply(clean_cwe)

# Showing result
print("Original CWE:\n", train_df['cwe_id'].head())
print("\nProcessed CWE:\n", train_df['cwe_clean'].head())

Original CWE:
 0    NVD-CWE-Other
1          CWE-732
2       ['CWE-20']
3          CWE-476
4      ['CWE-264']
Name: cwe_id, dtype: str

Processed CWE:
 0      0
1    732
2     20
3    476
4    264
Name: cwe_clean, dtype: int64


## Data Preapation for model

In [88]:
# Creating TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=5000)

# Fitting on training data and transform
X_train_code = tfidf.fit_transform(train_df['clean_code'])

# Only transform test data
X_test_code = tfidf.transform(test_df['clean_code'])

# Checking shapes
print("Train Code Feature Shape:", X_train_code.shape)
print("Test Code Feature Shape:", X_test_code.shape)

Train Code Feature Shape: (10826, 5000)
Test Code Feature Shape: (2706, 5000)


In [89]:
# Taking cwe as numeric feature
X_train_cwe = train_df[['cwe_clean']].values
X_test_cwe = test_df[['cwe_clean']].values

In [90]:
# Combining code features (TF-IDF) + cwe
X_train = hstack([X_train_code, X_train_cwe])
X_test = hstack([X_test_code, X_test_cwe])


In [91]:
# Target variable
y_train = train_df['target']
y_test = test_df['target']

In [92]:
# Checking shapes
print("Final Train Shape:", X_train.shape)
print("Final Test Shape:", X_test.shape)

Final Train Shape: (10826, 5001)
Final Test Shape: (2706, 5001)


## Model building

In [93]:
lr = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Storing models in dictionary
models = {
    "Logistic Regression": lr,
    "Random Forest": rf,
    "XGBoost": xgb
}

# Performing cross-validation using Recall
print("Cross Validation Results (Recall):\n")

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='recall')
    print(name, "Average Recall:", round(scores.mean(), 4))

Cross Validation Results (Recall):

Logistic Regression Average Recall: 0.8722
Random Forest Average Recall: 0.862


c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [06:08:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [06:08:32] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [06:08:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [06:08:43] WARNING: C:\a

XGBoost Average Recall: 0.9006


In [94]:
# Training all models on full training data
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# Getting probability predictions on test data
lr_prob = lr.predict_proba(X_test)[:, 1]
rf_prob = rf.predict_proba(X_test)[:, 1]
xgb_prob = xgb.predict_proba(X_test)[:, 1]

threshold = 0.3

# Converting probabilities to class (0 or 1)
lr_pred = (lr_prob > threshold).astype(int)
rf_pred = (rf_prob > threshold).astype(int)
xgb_pred = (xgb_prob > threshold).astype(int)



c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [06:09:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


## Model evaluation

In [95]:
# Function to evaluate model
def evaluate(name, y_true, y_pred):
    print("\n", name)
    print("Recall:", round(recall_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred), 4))
    print("F1 Score:", round(f1_score(y_true, y_pred), 4))


# Printing results
print("----- Test Evaluation -----")

evaluate("Logistic Regression", y_test, lr_pred)
evaluate("Random Forest", y_test, rf_pred)
evaluate("XGBoost", y_test, xgb_pred)

----- Test Evaluation -----

 Logistic Regression
Recall: 0.9512
Precision: 0.7513
F1 Score: 0.8395

 Random Forest
Recall: 0.9534
Precision: 0.7983
F1 Score: 0.869

 XGBoost
Recall: 0.9372
Precision: 0.8775
F1 Score: 0.9064


In [96]:
def predict(sample):
   
    code = sample.get("code", "")
    cwe = sample.get("cwe_id", "")
    
    
    code = str(code).lower()
    code = re.sub(r'//.*|/\*.*?\*/|#.*', '', code)
    code = re.sub(r'\s+', ' ', code).strip()
    
    
    numbers = re.findall(r'\d+', str(cwe))
    if len(numbers) > 0:
        cwe_clean = int(numbers[0])
    else:
        cwe_clean = 0
    
    
    code_vec = tfidf.transform([code])
    
   
    X = hstack([code_vec, [[cwe_clean]]])
    
    
    prob = xgb.predict_proba(X)[0][1]
    
    threshold = 0.3
    if prob > threshold:
        label = 1   # vulnerable
    else:
        label = 0   # safe
    
    return prob, label

In [97]:
# Take inputs one by one
cwe = input("Enter CWE ID  ")
code = input("Enter code snippet ")
repo = input("Enter repo name  ")
path = input("Enter file path ")
url = input("Enter URL  ")
sha = input("Enter SHA ")

# Creating dictionary of all inputs
sample = {
    "code": code,
    "cwe_id": cwe,
    "repo": repo,
    "path": path,
    "url": url,
    "sha": sha
}

prob, label = predict(sample)

# Showing result
print("\nPrediction Probability:", prob)

if label == 1:
    print(" Real Vulnerability")
else:
    print(" False Positive")


Prediction Probability: 0.95396
 Real Vulnerability


# END OF NOTEBOOK


In [98]:
import pickle

pickle.dump(tfidf, open("tfidf.pkl", "wb"))
pickle.dump(xgb, open("model.pkl", "wb"))